# SAE-XCrash — NB01: Data Protocol v2
**Work Package WP1 — Addendum** | P8: Target Encoding for High-Cardinality Features

**Load-and-continue** notebook. Does NOT rerun preprocessing.
Loads the already-processed train/val/test parquets and the raw US-Accidents data,
computes target-encoded street and city features, and adds them to the parquets.

All outputs go to `logs/wp1_v2/`. Processed parquets are updated in-place with
two new columns (`s_street_te`, `s_city_te`). Original label-encoded columns
(`s_street_enc`, `s_city_enc`) are preserved for reference.

Run order: NB01 (original) first. Then this notebook.


---
## Step 1 — Environment & Paths

In [1]:
!pip install -q pyarrow pandas numpy

import os, json
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

PROJ_ROOT  = Path('/content/drive/MyDrive/SAE-XCrash')
RAW_DIR    = PROJ_ROOT / 'data' / 'raw'
PROC_DIR   = PROJ_ROOT / 'data' / 'processed'
LOGS_DIR_V1= PROJ_ROOT / 'logs'

LOGS_DIR_V2 = PROJ_ROOT / 'logs' / 'wp1_v2'
LOGS_DIR_V2.mkdir(parents=True, exist_ok=True)

SEED      = 42
SMOOTH_K  = 10     # additive smoothing constant

print(f"Project root : {PROJ_ROOT}")
print(f"Saving to    : {LOGS_DIR_V2}")


Mounted at /content/drive
Project root : /content/drive/MyDrive/SAE-XCrash
Saving to    : /content/drive/MyDrive/SAE-XCrash/logs/wp1_v2


---
## Step 2 — Load Processed Parquets
Load the already-computed train/val/test splits. We need the `label` column (from training split) to fit encoding maps.

In [2]:
print("Loading processed train/val/test splits ...")
usa_train = pd.read_parquet(PROC_DIR / 'usa_train_processed.parquet')
usa_val   = pd.read_parquet(PROC_DIR / 'usa_val_processed.parquet')
usa_test  = pd.read_parquet(PROC_DIR / 'usa_test_processed.parquet')
print(f"  Train: {len(usa_train):,}  Val: {len(usa_val):,}  Test: {len(usa_test):,}")

global_mean = float(usa_train['label'].mean())
print(f"  Global severe rate (train): {global_mean:.4f}")

# Check if raw string columns (Street, City) survived into processed parquets
for col in ['Street', 'City', 's_street_enc', 's_city_enc']:
    present = col in usa_train.columns
    print(f"  '{col}' in train parquet: {present}")


Loading processed train/val/test splits ...
  Train: 5,549,870  Val: 1,268,806  Test: 166,552
  Global severe rate (train): 0.0271
  'Street' in train parquet: False
  'City' in train parquet: False
  's_street_enc' in train parquet: False
  's_city_enc' in train parquet: False


---
## Step 3 — Load Raw Data (if needed)
If `Street` and `City` columns are not in the processed parquet, load them from the raw US-Accidents CSV and join by a key column.

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 3 — Locate raw US-Accidents data
# Searches common locations and filenames; falls back to frequency encoding
# if the raw file cannot be found (frequency encoding is the safe alternative).
# ─────────────────────────────────────────────────────────────────────────────
import glob as _glob

STREET_IN_PROC = 'Street' in usa_train.columns
CITY_IN_PROC   = 'City'   in usa_train.columns

if STREET_IN_PROC and CITY_IN_PROC:
    print("Street and City columns found in processed parquets — no raw load needed ✓")
    df_train_raw   = usa_train[['Street','City','label']].copy()
    df_val_street  = usa_val  [['Street','City']].copy()
    df_test_street = usa_test [['Street','City']].copy()
    RAW_FOUND = True

else:
    print("Street / City not in processed parquets — searching for raw file ...")

    # ── Search broadly across common locations and name patterns ───────────────
    search_dirs = [
        PROJ_ROOT / 'data' / 'raw',
        PROJ_ROOT / 'data',
        PROJ_ROOT,
        Path('/content/drive/MyDrive'),
        Path('/content/drive/MyDrive/SAE-XCrash'),
        Path('/content'),
    ]
    name_patterns = [
        'US_Accidents*.csv', 'us_accidents*.csv', 'US_accidents*.csv',
        'US_Accidents*.parquet', 'us_accidents*.parquet',
        'USA_Accidents*.csv', 'accidents*.csv',
        '*Accidents*2023*.csv', '*accidents*.csv',
    ]

    raw_candidates = []
    for d in search_dirs:
        if d.exists():
            for pat in name_patterns:
                raw_candidates.extend(d.glob(pat))
            # Also check one level deeper
            for sub in d.iterdir():
                if sub.is_dir():
                    for pat in name_patterns:
                        raw_candidates.extend(sub.glob(pat))

    # Deduplicate
    raw_candidates = list(dict.fromkeys(raw_candidates))

    if raw_candidates:
        # Prefer largest file (most likely the full dataset)
        raw_path = max(raw_candidates, key=lambda p: p.stat().st_size)
        print(f"  Found: {raw_path}  ({raw_path.stat().st_size / 1e9:.2f} GB)")
        RAW_FOUND = True
    else:
        print("  Raw file not found in any searched location.")
        print("  Searched:")
        for d in search_dirs:
            print(f"    {d}  (exists={d.exists()})")
        # List what IS in the raw dir (so user can identify the correct name)
        raw_dir = PROJ_ROOT / 'data' / 'raw'
        if raw_dir.exists():
            files = list(raw_dir.iterdir())
            print(f"\n  Contents of {raw_dir}:")
            for f in files[:20]:
                size = f.stat().st_size / 1e6 if f.is_file() else 0
                print(f"    {f.name}  ({size:.1f} MB)" if f.is_file() else f"    {f.name}/")
        RAW_FOUND = False

    if RAW_FOUND:
        # ── Load only the columns we need ─────────────────────────────────────
        print(f"  Loading Street + City columns ...")
        needed_cols = ['ID','Street','City']

        if str(raw_path).endswith('.csv'):
            # Peek at header first
            import pandas as pd
            header = pd.read_csv(raw_path, nrows=0).columns.tolist()
            available = [c for c in needed_cols if c in header]
            print(f"  Available needed cols: {available}")
            raw_df = pd.read_csv(raw_path, usecols=available, low_memory=False)
        else:
            import pyarrow.parquet as pq
            schema_cols = pq.read_schema(raw_path).names
            available = [c for c in needed_cols if c in schema_cols]
            raw_df = pd.read_parquet(raw_path, columns=available)

        print(f"  Loaded {len(raw_df):,} rows")
        print(f"  Street NaN: {raw_df['Street'].isna().mean()*100:.1f}%  "
              f"City NaN: {raw_df['City'].isna().mean()*100:.1f}%")

        # ── Join to splits by ID ───────────────────────────────────────────────
        for split_name, split_df, out_name in [
            ('train', usa_train, 'df_train_raw'),
            ('val',   usa_val,   'df_val_street'),
            ('test',  usa_test,  'df_test_street'),
        ]:
            if 'ID' in split_df.columns and 'ID' in raw_df.columns:
                merged = split_df[['ID']].merge(raw_df[['ID','Street','City']],
                                                 on='ID', how='left')
                if split_name == 'train':
                    merged['label'] = split_df['label'].values
                    df_train_raw   = merged[['Street','City','label']]
                elif split_name == 'val':
                    df_val_street  = merged[['Street','City']]
                else:
                    df_test_street = merged[['Street','City']]
            else:
                print(f"  WARNING: ID not available for {split_name} — "
                      f"using positional join (requires same sort order)")
                if len(raw_df) >= len(split_df):
                    # Not safe — fall back
                    RAW_FOUND = False
                    break

if not RAW_FOUND:
    print("\n" + "="*60)
    print("FALLBACK: Using frequency encoding instead of target encoding.")
    print("Frequency encoding = log(count_in_train) — does not require label")
    print("and avoids the ordinality problem of label encoding.")
    print("This is an acceptable alternative; update Appendix F accordingly.")
    print("="*60)

    # ── Frequency encoding from label-encoded codes already in parquet ────────
    # We use value counts of s_state as a proxy indicator that this works
    # For street/city: we don't have raw strings, so we encode the label code
    # using its training-set frequency (log scale, normalised)
    for col_src, col_out in [('s_geohash5_code','s_street_te'),
                               ('s_geohash6_code','s_city_te')]:
        # Frequency encode from the spatial code as a stand-in
        # (geohash5 ≈ street-area; geohash6 ≈ block-level)
        if col_src in usa_train.columns:
            freq_map = np.log1p(usa_train[col_src].value_counts()).to_dict()
            for split_name, split_df in [('train',usa_train),('val',usa_val),('test',usa_test)]:
                split_df[col_out] = split_df[col_src].map(freq_map).fillna(0).astype('float32')
            print(f"  Created {col_out} from {col_src} frequency encoding")
        else:
            for split_df in [usa_train, usa_val, usa_test]:
                split_df[col_out] = np.float32(0.0)
            print(f"  {col_src} not found — {col_out} set to 0 (placeholder)")

    # Mark that we used fallback so Step 4 is skipped
    ENCODING_METHOD = 'frequency'
else:
    ENCODING_METHOD = 'target'

print(f"\nEncoding method: {ENCODING_METHOD}")
print("Proceed to Step 4 ✓")


Street / City not in processed parquets — searching for raw file ...
  Found: /content/drive/MyDrive/SAE-XCrash/data/raw/us_accidents/US_Accidents_March23.csv  (3.06 GB)
  Loading Street + City columns ...
  Available needed cols: ['ID', 'Street', 'City']
  Loaded 7,728,394 rows
  Street NaN: 0.1%  City NaN: 0.0%

Encoding method: target
Proceed to Step 4 ✓


---
## Step 4 — Compute and Apply Target Encoding
Fit encoding maps on training split only (2016–2021). Apply to val and test with global-mean fallback for unseen categories.

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# P8 — Target Encoding with Additive Smoothing
# enc(category) = (sum_target + K * global_mean) / (count + K)
# Unseen categories → global_mean
# ─────────────────────────────────────────────────────────────────────────────
print(f"Target encoding (smooth_k={SMOOTH_K}, global_mean={global_mean:.4f})")
print("=" * 60)

def fit_target_encoding(cat_series, label_series, smooth_k, global_mean):
    """Fit encoding map on training data."""
    df = pd.DataFrame({'cat': cat_series.fillna('__MISSING__'), 'label': label_series})
    stats = df.groupby('cat')['label'].agg(['sum','count'])
    stats['encoded'] = (stats['sum'] + smooth_k * global_mean) / (stats['count'] + smooth_k)
    return stats['encoded'].to_dict()

def apply_target_encoding(cat_series, mapping, global_mean):
    """Apply encoding map; unseen → global_mean."""
    return cat_series.fillna('__MISSING__').map(mapping).fillna(global_mean).astype('float32')

# ── Street encoding ────────────────────────────────────────────────────────────
street_map = fit_target_encoding(df_train_raw['Street'], df_train_raw['label'],
                                  SMOOTH_K, global_mean)
print(f"  Street: {len(street_map):,} unique categories")
print(f"  Range  : [{min(street_map.values()):.4f}, {max(street_map.values()):.4f}]")

usa_train['s_street_te'] = apply_target_encoding(df_train_raw['Street'], street_map, global_mean)
usa_val  ['s_street_te'] = apply_target_encoding(df_val_street['Street'],  street_map, global_mean)
usa_test ['s_street_te'] = apply_target_encoding(df_test_street['Street'], street_map, global_mean)
print(f"  Train mean: {usa_train['s_street_te'].mean():.4f}  "
      f"Val mean: {usa_val['s_street_te'].mean():.4f}  "
      f"Test mean: {usa_test['s_street_te'].mean():.4f}")

# ── City encoding ──────────────────────────────────────────────────────────────
city_map = fit_target_encoding(df_train_raw['City'], df_train_raw['label'],
                                SMOOTH_K, global_mean)
print(f"\n  City  : {len(city_map):,} unique categories")
print(f"  Range  : [{min(city_map.values()):.4f}, {max(city_map.values()):.4f}]")

usa_train['s_city_te'] = apply_target_encoding(df_train_raw['City'], city_map, global_mean)
usa_val  ['s_city_te'] = apply_target_encoding(df_val_street['City'],  city_map, global_mean)
usa_test ['s_city_te'] = apply_target_encoding(df_test_street['City'], city_map, global_mean)
print(f"  Train mean: {usa_train['s_city_te'].mean():.4f}  "
      f"Val mean: {usa_val['s_city_te'].mean():.4f}  "
      f"Test mean: {usa_test['s_city_te'].mean():.4f}")

# ── Leakage check: train encodings should ≈ global mean ───────────────────────
# If city/street means differ a lot from global_mean, check for memorization
print(f"\n  Global mean           : {global_mean:.4f}")
print(f"  s_city_te  train mean : {usa_train['s_city_te'].mean():.4f} (should ≈ global_mean)")
print(f"  s_street_te train mean: {usa_train['s_street_te'].mean():.4f}")
print("  Note: slight deviation from global_mean is normal due to smoothing")

# ── Correlation with label ────────────────────────────────────────────────────
r_street = usa_train['s_street_te'].corr(usa_train['label'])
r_city   = usa_train['s_city_te'].corr(usa_train['label'])
print(f"\n  Pearson r with label (train):")
print(f"    s_street_te : {r_street:.4f}")
print(f"    s_city_te   : {r_city:.4f}")
print("  (expected ~0.03-0.15; higher = feature captures severity heterogeneity)")


Target encoding (smooth_k=10, global_mean=0.0271)
  Street: 268,842 unique categories
  Range  : [0.0001, 0.8987]
  Train mean: 0.0250  Val mean: 0.0314  Test mean: 0.0333

  City  : 13,093 unique categories
  Range  : [0.0002, 0.7744]
  Train mean: 0.0258  Val mean: 0.0310  Test mean: 0.0322

  Global mean           : 0.0271
  s_city_te  train mean : 0.0258 (should ≈ global_mean)
  s_street_te train mean: 0.0250
  Note: slight deviation from global_mean is normal due to smoothing

  Pearson r with label (train):
    s_street_te : 0.3804
    s_city_te   : 0.2791
  (expected ~0.03-0.15; higher = feature captures severity heterogeneity)


---
## Step 5 — Save Updated Parquets and Encoding Maps

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Save updated parquets (with s_street_te, s_city_te added)
# and encoding metadata for reproducibility
# ─────────────────────────────────────────────────────────────────────────────

# Save updated parquets
for name, df in [('train', usa_train), ('val', usa_val), ('test', usa_test)]:
    path = PROC_DIR / f'usa_{name}_processed.parquet'
    df.to_parquet(path, index=False)
    new_cols = [c for c in df.columns if c.endswith('_te')]
    print(f"  Updated usa_{name}_processed.parquet  (new cols: {new_cols})")

# Save encoding maps + metadata
te_meta = {
    'smooth_k': SMOOTH_K,
    'global_mean': round(global_mean, 6),
    'n_street_categories': len(street_map),
    'n_city_categories': len(city_map),
    'features_added': ['s_street_te', 's_city_te'],
    'generated_at': str(datetime.now()),
    'note': 'Maps fitted on training split (2016-2021) only. Unseen → global_mean.',
}
with open(LOGS_DIR_V2 / 'target_encoding_meta.json', 'w') as f:
    json.dump(te_meta, f, indent=2)
print(f"  Saved: target_encoding_meta.json")

# Save maps themselves (for future use / validation)
# Convert float32 values to float for JSON serialisation
with open(LOGS_DIR_V2 / 'street_encoding_map.json', 'w') as f:
    json.dump({k: round(float(v), 6) for k, v in street_map.items()}, f)
with open(LOGS_DIR_V2 / 'city_encoding_map.json', 'w') as f:
    json.dump({k: round(float(v), 6) for k, v in city_map.items()}, f)
print(f"  Saved: street_encoding_map.json, city_encoding_map.json")

print("\nNB01_Data_Protocol_v2 complete ✓")
print()
print("NEXT STEPS:")
print("  1. Run NB02_Modeling_Baselines_v2 — it will pick up s_street_te and")
print("     s_city_te automatically via the s_ prefix filter in FEAT_USA.")
print("  2. Compare AUPRC with vs. without target encoding (see ablation).")
print("  3. Update Appendix F and Section 4 in paper with target encoding description.")


  Updated usa_train_processed.parquet  (new cols: ['s_street_te', 's_city_te'])
  Updated usa_val_processed.parquet  (new cols: ['s_street_te', 's_city_te'])
  Updated usa_test_processed.parquet  (new cols: ['s_street_te', 's_city_te'])
  Saved: target_encoding_meta.json
  Saved: street_encoding_map.json, city_encoding_map.json

NB01_Data_Protocol_v2 complete ✓

NEXT STEPS:
  1. Run NB02_Modeling_Baselines_v2 — it will pick up s_street_te and
     s_city_te automatically via the s_ prefix filter in FEAT_USA.
  2. Compare AUPRC with vs. without target encoding (see ablation).
  3. Update Appendix F and Section 4 in paper with target encoding description.
